In [1]:
# ===========================================
# GNN-BASED DISASTER RESPONSE ROUTING SYSTEM
# Uses Graph Neural Networks for intelligent route planning
# ===========================================

!pip install geopandas osmnx networkx folium geopy torch torch-geometric --quiet

import geopandas as gpd
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np
from shapely.geometry import Point, LineString
from geopy.geocoders import Nominatim
import folium
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from torch_geometric.utils import from_networkx
from sklearn.preprocessing import StandardScaler
from datetime import datetime

# ===========================================
# CONFIGURATION
# ===========================================
TARGET_CITY = "Rudraprayag, Uttarakhand, India"
SAFE_DISTANCE_FROM_WATER = 500  # meters
np.random.seed(42)
torch.manual_seed(42)

print("="*70)
print("🧠 GNN-BASED DISASTER RESPONSE ROUTING SYSTEM")
print("="*70)

# ===========================================
# MODULE 1: GRAPH NEURAL NETWORK ARCHITECTURE
# ===========================================
class RoadBlockageGNN(nn.Module):
    """
    Graph Neural Network for predicting road blockage probability
    Uses Graph Attention Networks (GAT) for spatial reasoning
    """

    def __init__(self, num_node_features, hidden_channels=64):
        super(RoadBlockageGNN, self).__init__()

        # Graph Attention layers for spatial feature learning
        self.conv1 = GATConv(num_node_features, hidden_channels, heads=4, dropout=0.3)
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels, heads=4, dropout=0.3)
        self.conv3 = GATConv(hidden_channels * 4, hidden_channels, heads=2, dropout=0.3)

        # Edge prediction layers
        self.edge_predictor = nn.Sequential(
            nn.Linear(hidden_channels * 2 * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()  # Probability of blockage
        )

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # Node feature learning through attention layers
        x = F.elu(self.conv1(x, edge_index))
        x = F.elu(self.conv2(x, edge_index))
        x = F.elu(self.conv3(x, edge_index))

        # Store node embeddings
        self.node_embeddings = x

        return x

    def predict_edge_blockage(self, edge_index):
        """Predict blockage probability for each edge"""
        # Get embeddings for source and target nodes
        src_embeddings = self.node_embeddings[edge_index[0]]
        dst_embeddings = self.node_embeddings[edge_index[1]]

        # Concatenate source and target embeddings
        edge_embeddings = torch.cat([src_embeddings, dst_embeddings], dim=1)

        # Predict blockage probability
        blockage_prob = self.edge_predictor(edge_embeddings)

        return blockage_prob.squeeze()

class RoutePlanningGNN(nn.Module):
    """
    GNN for learning optimal routing decisions
    Predicts edge weights based on disaster context
    """

    def __init__(self, num_node_features, num_edge_features, hidden_channels=64):
        super(RoutePlanningGNN, self).__init__()

        # Node feature processing
        self.node_encoder = nn.Sequential(
            nn.Linear(num_node_features, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Graph convolution layers
        self.conv1 = GCNConv(hidden_channels, hidden_channels * 2)
        self.conv2 = GCNConv(hidden_channels * 2, hidden_channels)

        # Edge weight predictor
        self.edge_weight_predictor = nn.Sequential(
            nn.Linear(hidden_channels * 2 + num_edge_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Softplus()  # Ensures positive weights
        )

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        # Encode node features
        x = self.node_encoder(x)

        # Graph convolutions
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        self.node_embeddings = x

        return x

    def predict_edge_weights(self, edge_index, edge_attr):
        """Predict dynamic edge weights for routing"""
        src_embeddings = self.node_embeddings[edge_index[0]]
        dst_embeddings = self.node_embeddings[edge_index[1]]

        # Combine node embeddings and edge features
        edge_features = torch.cat([src_embeddings, dst_embeddings, edge_attr], dim=1)

        # Predict weights
        weights = self.edge_weight_predictor(edge_features)

        return weights.squeeze()

# ===========================================
# MODULE 2: DISASTER DETECTION & MONITORING
# ===========================================
class DisasterMonitor:
    """Enhanced disaster monitoring with GNN integration"""

    def __init__(self, location):
        self.location = location
        self.geolocator = Nominatim(user_agent="disaster_gnn_system")

    def get_location_coords(self):
        """Get coordinates for location"""
        try:
            location = self.geolocator.geocode(self.location)
            return (location.latitude, location.longitude)
        except:
            return None

    def detect_active_disasters(self):
        """Detect active disasters with rich feature extraction"""
        print(f"\n🔍 Scanning for active disasters near {self.location}...")

        disasters = []

        # Simulate disaster detection (replace with real API in production)
        if np.random.random() > 0.2:  # 80% chance of disaster for demo
            disaster_type = np.random.choice(['flood', 'earthquake', 'landslide'],
                                           p=[0.6, 0.25, 0.15])

            disasters.append({
                'type': disaster_type,
                'severity': np.random.choice(['low', 'medium', 'high'],
                                           p=[0.2, 0.5, 0.3]),
                'epicenter': self.get_location_coords(),
                'affected_radius_km': np.random.uniform(3, 10),
                'timestamp': datetime.now(),
                'intensity_score': np.random.uniform(5, 9),
                'source': self._get_disaster_source(disaster_type),
                'weather_condition': np.random.choice(['heavy_rain', 'normal', 'clear']),
                'time_since_start_hours': np.random.uniform(1, 12)
            })

        if len(disasters) > 0:
            print(f"⚠️  ACTIVE DISASTERS DETECTED: {len(disasters)}")
            for d in disasters:
                print(f"   • {d['type'].upper()} - Severity: {d['severity']}")
                print(f"     Intensity: {d['intensity_score']:.1f}/10")
                print(f"     Source: {d['source']}")
        else:
            print("✅ No active disasters detected")

        return disasters

    def _get_disaster_source(self, disaster_type):
        """Get realistic disaster source"""
        sources = {
            'flood': 'Mandakini River overflow',
            'earthquake': 'Himalayan tectonic activity',
            'landslide': 'Heavy rainfall on steep terrain'
        }
        return sources.get(disaster_type, 'Unknown source')

# ===========================================
# MODULE 3: GRAPH DATA PREPARATION
# ===========================================
class GraphDataPreparator:
    """Prepares road network as PyTorch Geometric graph"""

    def __init__(self, road_network, water_bodies, disasters):
        self.G_nx = road_network
        self.water_bodies = water_bodies
        self.disasters = disasters
        self.scaler = StandardScaler()
        # Create mapping from original node IDs to sequential integers
        self.node_to_idx = {node: idx for idx, node in enumerate(self.G_nx.nodes())}
        self.idx_to_node = {idx: node for node, idx in self.node_to_idx.items()}

    def prepare_graph_data(self):
        """Convert NetworkX graph to PyTorch Geometric format with features"""
        print(f"\n🔄 Converting to GNN-compatible graph structure...")

        # Extract node and edge features
        node_features = self._extract_node_features()
        edge_features = self._extract_edge_features()
        edge_labels = self._generate_edge_labels()

        # Convert to PyTorch Geometric Data object with mapped indices
        edges = []
        for u, v in self.G_nx.edges():
            # Map original node IDs to sequential indices
            u_idx = self.node_to_idx[u]
            v_idx = self.node_to_idx[v]
            edges.append([u_idx, v_idx])

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

        # Add reverse edges for undirected graph
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

        # Duplicate edge features and labels for bidirectional edges
        edge_features = torch.cat([edge_features, edge_features], dim=0)
        edge_labels = torch.cat([edge_labels, edge_labels], dim=0)

        # Verify edge indices are valid
        max_idx = edge_index.max().item()
        num_nodes = len(self.G_nx.nodes())
        if max_idx >= num_nodes:
            print(f"❌ Error: Edge index {max_idx} >= num_nodes {num_nodes}")
            raise ValueError(f"Invalid edge index detected")

        # Create PyG Data object
        data = Data(
            x=node_features,
            edge_index=edge_index,
            edge_attr=edge_features,
            edge_labels=edge_labels,
            num_nodes=num_nodes
        )

        print(f"✅ Graph prepared: {data.num_nodes} nodes, {data.edge_index.shape[1]} edges")
        print(f"   Node features: {data.x.shape[1]} dimensions")
        print(f"   Edge features: {data.edge_attr.shape[1]} dimensions")
        print(f"   Edge index range: [0, {max_idx}]")

        return data

    def _extract_node_features(self):
        """Extract features for each node (intersection)"""
        features = []

        nodes_gdf = ox.graph_to_gdfs(self.G_nx, edges=False, nodes=True)
        nodes_gdf = nodes_gdf.to_crs(epsg=3857)

        # Iterate in the same order as node_to_idx mapping
        for node_id in self.G_nx.nodes():
            node_data = self.G_nx.nodes[node_id]
            node_features = []

            # Spatial features
            node_features.append(node_data['y'])  # Latitude
            node_features.append(node_data['x'])  # Longitude

            # Degree centrality
            node_features.append(self.G_nx.degree(node_id))

            # Distance to nearest water body
            if len(self.water_bodies) > 0:
                try:
                    water_gdf = self.water_bodies.to_crs(epsg=3857)
                    node_point = nodes_gdf.loc[node_id, 'geometry']
                    min_water_dist = water_gdf.distance(node_point).min()
                    node_features.append(min_water_dist)
                except:
                    node_features.append(5000)
            else:
                node_features.append(5000)  # Default 5km

            # Distance to disaster epicenter
            if len(self.disasters) > 0:
                epicenter = self.disasters[0]['epicenter']
                if epicenter:
                    dist = self._calculate_distance(
                        (node_data['y'], node_data['x']),
                        epicenter
                    )
                    node_features.append(dist)
                else:
                    node_features.append(10000)
            else:
                node_features.append(10000)

            # Elevation (simulated - replace with real DEM data)
            node_features.append(np.random.uniform(500, 2000))

            features.append(node_features)

        # Normalize features
        features_array = np.array(features)
        features_normalized = self.scaler.fit_transform(features_array)

        return torch.tensor(features_normalized, dtype=torch.float)

    def _extract_edge_features(self):
        """Extract features for each edge (road segment)"""
        features = []

        edges_gdf = ox.graph_to_gdfs(self.G_nx, nodes=False, edges=True)
        edges_gdf = edges_gdf.to_crs(epsg=3857)

        for u, v, data in self.G_nx.edges(data=True):
            edge_features = []

            # Physical features
            edge_features.append(data.get('length', 100))  # Length in meters

            # Road type encoding (simplified)
            highway_type = data.get('highway', 'residential')
            if isinstance(highway_type, list):
                highway_type = highway_type[0]

            road_type_encoding = {
                'motorway': 5, 'trunk': 4, 'primary': 3,
                'secondary': 2, 'tertiary': 1, 'residential': 0
            }
            edge_features.append(road_type_encoding.get(highway_type, 0))

            # Number of lanes (estimated)
            lanes = data.get('lanes', 2)
            if isinstance(lanes, list):
                lanes = int(lanes[0]) if lanes[0].isdigit() else 2
            elif isinstance(lanes, str):
                lanes = int(lanes) if lanes.isdigit() else 2
            edge_features.append(lanes)

            # Distance to water (for flood risk)
            if len(self.water_bodies) > 0:
                try:
                    water_gdf = self.water_bodies.to_crs(epsg=3857)
                    edge_geom = edges_gdf.loc[(u, v, 0), 'geometry']
                    min_water_dist = water_gdf.distance(edge_geom).min()
                    edge_features.append(min_water_dist)
                except:
                    edge_features.append(5000)
            else:
                edge_features.append(5000)

            # Disaster context features
            if len(self.disasters) > 0:
                disaster = self.disasters[0]
                edge_features.append(disaster['intensity_score'])

                # Disaster type encoding
                disaster_encoding = {'flood': 0, 'earthquake': 1, 'landslide': 2}
                edge_features.append(disaster_encoding.get(disaster['type'], 0))

                # Time since disaster
                edge_features.append(disaster['time_since_start_hours'])
            else:
                edge_features.extend([0, 0, 0])

            features.append(edge_features)

        return torch.tensor(features, dtype=torch.float)

    def _generate_edge_labels(self):
        """Generate ground truth labels for edge blockage"""
        labels = []

        edges_gdf = ox.graph_to_gdfs(self.G_nx, nodes=False, edges=True)
        edges_gdf = edges_gdf.to_crs(epsg=3857)

        for u, v, data in self.G_nx.edges(data=True):
            # Simulate blockage based on disaster type and proximity
            is_blocked = 0

            if len(self.disasters) > 0:
                disaster = self.disasters[0]

                if disaster['type'] == 'flood':
                    # Check proximity to water
                    if len(self.water_bodies) > 0:
                        try:
                            water_gdf = self.water_bodies.to_crs(epsg=3857)
                            edge_geom = edges_gdf.loc[(u, v, 0), 'geometry']
                            min_water_dist = water_gdf.distance(edge_geom).min()

                            if disaster['severity'] == 'high' and min_water_dist < SAFE_DISTANCE_FROM_WATER * 2:
                                is_blocked = 1
                            elif disaster['severity'] == 'medium' and min_water_dist < SAFE_DISTANCE_FROM_WATER:
                                is_blocked = 1
                            elif disaster['severity'] == 'low' and min_water_dist < SAFE_DISTANCE_FROM_WATER * 0.5:
                                is_blocked = 1
                        except:
                            pass

                elif disaster['type'] == 'earthquake':
                    # Random blockage based on severity
                    severity_prob = {'low': 0.05, 'medium': 0.15, 'high': 0.30}
                    if np.random.random() < severity_prob.get(disaster['severity'], 0.1):
                        is_blocked = 1

                elif disaster['type'] == 'landslide':
                    # Blockage based on terrain (simulated)
                    if np.random.random() < 0.25:
                        is_blocked = 1

            labels.append(is_blocked)

        return torch.tensor(labels, dtype=torch.float)

    def _calculate_distance(self, coord1, coord2):
        """Calculate distance between two coordinates in meters"""
        from geopy.distance import geodesic
        return geodesic(coord1, coord2).meters

# ===========================================
# MODULE 4: GNN TRAINING & INFERENCE
# ===========================================
class GNNRoutingSystem:
    """Manages GNN training and inference for routing"""

    def __init__(self, graph_data):
        self.data = graph_data
        self.blockage_model = None
        self.routing_model = None

    def train_blockage_model(self, epochs=50):
        """Train GNN to predict road blockages"""
        print(f"\n🧠 Training Road Blockage Prediction GNN...")

        # Initialize model
        num_node_features = self.data.x.shape[1]
        self.blockage_model = RoadBlockageGNN(num_node_features)

        optimizer = torch.optim.Adam(self.blockage_model.parameters(),
                                    lr=0.01, weight_decay=5e-4)
        criterion = nn.BCELoss()

        # Training loop
        self.blockage_model.train()
        losses = []

        for epoch in range(epochs):
            optimizer.zero_grad()

            # Forward pass
            _ = self.blockage_model(self.data)
            blockage_pred = self.blockage_model.predict_edge_blockage(
                self.data.edge_index
            )

            # Calculate loss
            loss = criterion(blockage_pred, self.data.edge_labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

            if (epoch + 1) % 10 == 0:
                # Calculate accuracy
                pred_binary = (blockage_pred > 0.5).float()
                accuracy = (pred_binary == self.data.edge_labels).float().mean()
                print(f"   Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | Acc: {accuracy:.4f}")

        print(f"✅ Blockage model trained")
        return losses

    def predict_blocked_edges(self, threshold=0.5):
        """Predict which edges are blocked"""
        self.blockage_model.eval()

        with torch.no_grad():
            _ = self.blockage_model(self.data)
            blockage_prob = self.blockage_model.predict_edge_blockage(
                self.data.edge_index
            )

            # Get blocked edges (consider only one direction)
            num_original_edges = blockage_prob.shape[0] // 2
            blockage_prob_original = blockage_prob[:num_original_edges]

            blocked_mask = blockage_prob_original > threshold
            blocked_indices = torch.where(blocked_mask)[0]

        print(f"\n🚧 Predicted {blocked_indices.shape[0]} blocked road segments")

        return blocked_indices, blockage_prob_original

    def get_modified_graph_data(self, blocked_indices):
        """Create new graph with blocked edges removed"""
        # Get original edges (before doubling for bidirectional)
        num_original_edges = self.data.edge_index.shape[1] // 2

        # Create mask for non-blocked edges
        mask = torch.ones(num_original_edges, dtype=torch.bool)
        mask[blocked_indices] = False

        # Apply mask to edges and features
        edge_index_filtered = self.data.edge_index[:, :num_original_edges][:, mask]
        edge_attr_filtered = self.data.edge_attr[:num_original_edges][mask]

        # Make bidirectional again
        edge_index_filtered = torch.cat([edge_index_filtered, edge_index_filtered.flip(0)], dim=1)
        edge_attr_filtered = torch.cat([edge_attr_filtered, edge_attr_filtered], dim=0)

        # Create new data object
        filtered_data = Data(
            x=self.data.x,
            edge_index=edge_index_filtered,
            edge_attr=edge_attr_filtered,
            num_nodes=self.data.num_nodes
        )

        return filtered_data

# ===========================================
# MODULE 5: ROUTE PLANNING WITH GNN
# ===========================================
class GNNRoutePlanner:
    """Plans routes using GNN predictions"""

    def __init__(self, original_graph, filtered_data, blocked_indices):
        self.G_nx = original_graph
        self.filtered_data = filtered_data
        self.blocked_indices = blocked_indices
        self.geolocator = Nominatim(user_agent="disaster_gnn_system")

    def get_user_location(self):
        """Get user's current location"""
        print("\n" + "="*70)
        print("📍 USER LOCATION INPUT")
        print("="*70)
        print("\nOptions:")
        print("  1. Enter coordinates (latitude, longitude)")
        print("  2. Enter location name/address")
        print("  3. Use city center (default)")

        choice = input("\nSelect option (1/2/3): ").strip()

        if choice == '1':
            try:
                lat = float(input("Enter latitude: "))
                lon = float(input("Enter longitude: "))
                return (lat, lon)
            except:
                print("❌ Invalid coordinates. Using default.")
                return self._get_default_location()
        elif choice == '2':
            location_name = input("Enter location: ").strip()
            try:
                location = self.geolocator.geocode(location_name)
                if location:
                    print(f"✅ Found: {location.address}")
                    return (location.latitude, location.longitude)
            except:
                pass
            print("❌ Location not found. Using default.")
            return self._get_default_location()
        else:
            return self._get_default_location()

    def _get_default_location(self):
        """Get default location"""
        try:
            location = self.geolocator.geocode(TARGET_CITY)
            return (location.latitude, location.longitude)
        except:
            return (30.2857, 79.0746)

    def create_modified_networkx(self):
        """Create NetworkX graph without blocked edges"""
        G_modified = self.G_nx.copy()

        # Get edge list - use the same order as when we created the graph data
        edge_list = list(self.G_nx.edges())

        # Remove blocked edges using original node IDs
        for idx in self.blocked_indices.numpy():
            if idx < len(edge_list):
                u, v = edge_list[idx]
                try:
                    # Remove all edges between u and v (handles multi-edges)
                    if G_modified.has_edge(u, v):
                        G_modified.remove_edge(u, v)
                except:
                    pass

        print(f"🔄 Modified network: {len(G_modified.edges)} edges remaining")
        return G_modified

    def find_nearest_shelter(self, user_coords, shelters):
        """Find nearest reachable shelter using GNN-modified network"""
        print(f"\n🏥 Finding nearest shelter...")

        G_modified = self.create_modified_networkx()

        user_node = ox.distance.nearest_nodes(G_modified, user_coords[1], user_coords[0])

        shelter_distances = []
        for idx, shelter in shelters.iterrows():
            shelter_coords = (shelter.geometry.y, shelter.geometry.x)
            shelter_node = ox.distance.nearest_nodes(
                G_modified,
                shelter.geometry.x,
                shelter.geometry.y
            )

            try:
                route_length = nx.shortest_path_length(
                    G_modified,
                    user_node,
                    shelter_node,
                    weight='length'
                )

                shelter_distances.append({
                    'idx': idx,
                    'coords': shelter_coords,
                    'node': shelter_node,
                    'distance': route_length,
                    'name': shelter.get('name', f'Shelter {idx}'),
                    'graph': G_modified,
                    'original_graph': self.G_nx  # Store original for visualization
                })
            except nx.NetworkXNoPath:
                continue

        if len(shelter_distances) == 0:
            print("❌ No reachable shelters found")
            return None

        nearest = min(shelter_distances, key=lambda x: x['distance'])
        print(f"✅ Nearest shelter: {nearest['name']}")
        print(f"   Distance: {nearest['distance']:.0f} meters ({nearest['distance']/1000:.2f} km)")

        return nearest

    def calculate_route(self, user_coords, shelter_info):
        """Calculate optimal route"""
        print(f"\n🛣️  Calculating GNN-optimized route...")

        G_modified = shelter_info['graph']
        user_node = ox.distance.nearest_nodes(G_modified, user_coords[1], user_coords[0])
        shelter_node = shelter_info['node']

        try:
            route = nx.shortest_path(G_modified, user_node, shelter_node, weight='length')
            route_length = nx.shortest_path_length(G_modified, user_node, shelter_node, weight='length')
            travel_time = (route_length / 1000) / 30 * 60  # 30 km/h average

            print(f"✅ Route calculated")
            print(f"   Distance: {route_length/1000:.2f} km")
            print(f"   Est. time: {travel_time:.1f} minutes")
            print(f"   Segments: {len(route)-1}")

            return route, route_length, travel_time, G_modified
        except:
            print("❌ No path found")
            return None, None, None, None

# ===========================================
# MODULE 6: VISUALIZATION
# ===========================================
def visualize_route(G, route, user_coords, shelter_info, blocked_indices, shelters):
    """Create interactive map"""
    print(f"\n🗺️  Creating interactive map...")

    m = folium.Map(location=user_coords, zoom_start=14, tiles='OpenStreetMap')

    # User location
    folium.Marker(
        user_coords,
        popup='📍 Your Location',
        tooltip='Start Point',
        icon=folium.Icon(color='blue', icon='user', prefix='fa')
    ).add_to(m)

    # Shelter
    folium.Marker(
        shelter_info['coords'],
        popup=f"🏥 {shelter_info['name']}",
        tooltip='Emergency Shelter',
        icon=folium.Icon(color='green', icon='home', prefix='fa')
    ).add_to(m)

    # Other shelters
    for idx, shelter in shelters.iterrows():
        try:
            if idx != shelter_info['idx']:
                geom = shelter.geometry
                if geom.geom_type != 'Point':
                    geom = geom.centroid

                folium.Marker(
                    (geom.y, geom.x),
                    popup=f"🏥 {shelter.get('name', f'Shelter {idx}')}",
                    icon=folium.Icon(color='lightgray', icon='home', prefix='fa')
                ).add_to(m)
        except Exception as e:
            continue

    # Route
    if route:
        try:
            route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in route]
            folium.PolyLine(route_coords, color='blue', weight=6, opacity=0.8,
                           popup='🛣️ GNN-Optimized Safe Route').add_to(m)
        except Exception as e:
            print(f"   ⚠️  Could not draw route on map: {e}")

    # Blocked roads (sample) - use original edge list
    try:
        original_G = shelter_info.get('original_graph', G)
        edge_list = list(original_G.edges())

        blocked_count = 0
        for idx in blocked_indices.numpy()[:30]:  # Limit to 30 for performance
            if idx < len(edge_list):
                u, v = edge_list[idx]
                try:
                    coords = [(original_G.nodes[u]['y'], original_G.nodes[u]['x']),
                             (original_G.nodes[v]['y'], original_G.nodes[v]['x'])]
                    folium.PolyLine(coords, color='red', weight=3, opacity=0.6,
                              popup='⚠️ GNN Predicted Blockage').add_to(m)
                    blocked_count += 1
                except:
                    continue

        if blocked_count > 0:
            print(f"   ✅ Displayed {blocked_count} blocked roads on map")
    except Exception as e:
        print(f"   ⚠️  Could not display blocked roads: {e}")

    # Add legend
    legend_html = '''
    <div style="position: fixed;
                bottom: 50px; right: 50px; width: 220px; height: 200px;
                background-color: white; border:2px solid grey; z-index:9999;
                font-size:14px; padding: 10px; border-radius: 5px;">
    <p style="margin:0; font-weight:bold">🗺️ Map Legend</p>
    <hr style="margin: 5px 0;">
    <p style="margin:5px 0">🔵 Your Location</p>
    <p style="margin:5px 0">🟢 Target Shelter</p>
    <p style="margin:5px 0">⚪ Other Shelters</p>
    <p style="margin:5px 0"><span style="color:blue; font-weight:bold">━━</span> Safe Route</p>
    <p style="margin:5px 0"><span style="color:red; font-weight:bold">━━</span> Blocked Roads</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))

    map_file = 'gnn_disaster_route_map.html'
    m.save(map_file)
    print(f"✅ Map saved as '{map_file}'")
    return m

# ===========================================
# MAIN EXECUTION
# ===========================================
def main():
    """Main execution pipeline"""

    # Step 1: Monitor disasters
    monitor = DisasterMonitor(TARGET_CITY)
    disasters = monitor.detect_active_disasters()

    # Step 2: Load geographic data
    print(f"\n📍 Loading geographic data for {TARGET_CITY}...")
    try:
        road_network = ox.graph_from_place(TARGET_CITY, network_type='drive', simplify=True)
        print(f"✅ Loaded {len(road_network.nodes)} nodes, {len(road_network.edges)} edges")
    except:
        print("❌ Failed to load road network")
        return

    try:
        water_bodies = ox.features.features_from_place(
            TARGET_CITY,
            tags={'natural': ['water', 'waterway'], 'waterway': ['river', 'stream']}
        )
        print(f"✅ Loaded {len(water_bodies)} water features")
    except:
        print("⚠️  No water bodies found")
        water_bodies = gpd.GeoDataFrame()

    # Load shelters
    try:
        shelter_tags = {
            'amenity': ['community_centre', 'social_facility', 'school', 'hospital'],
            'building': ['civic', 'government']
        }
        shelters_raw = ox.features.features_from_place(TARGET_CITY, tags=shelter_tags)

        # Convert all geometries to points (centroids for polygons)
        shelters_data = []
        for idx, shelter in shelters_raw.iterrows():
            geom = shelter.geometry
            if geom.geom_type == 'Polygon' or geom.geom_type == 'MultiPolygon':
                point_geom = geom.centroid
            elif geom.geom_type == 'Point':
                point_geom = geom
            else:
                continue  # Skip other geometry types

            shelters_data.append({
                'geometry': point_geom,
                'name': shelter.get('name', f'Shelter {idx}'),
                'amenity': shelter.get('amenity', 'shelter')
            })

        shelters = gpd.GeoDataFrame(shelters_data, crs='EPSG:4326')
        print(f"✅ Loaded {len(shelters)} potential shelters")
    except:
        print("⚠️  Creating synthetic shelters")
        geolocator = Nominatim(user_agent="disaster_gnn")
        location = geolocator.geocode(TARGET_CITY)
        center_lat, center_lon = location.latitude, location.longitude

        shelters_data = []
        for i in range(5):
            angle = (2 * np.pi * i) / 5
            offset = 0.01
            shelters_data.append({
                'geometry': Point(center_lon + offset * np.cos(angle),
                                center_lat + offset * np.sin(angle)),
                'name': f'Emergency Shelter {i+1}',
                'capacity': np.random.randint(50, 200)
            })
        shelters = gpd.GeoDataFrame(shelters_data, crs='EPSG:4326')

    # Step 3: Prepare graph data for GNN
    print(f"\n{'='*70}")
    print("🔧 PREPARING GRAPH DATA FOR GNN")
    print(f"{'='*70}")

    data_prep = GraphDataPreparator(road_network, water_bodies, disasters)
    graph_data = data_prep.prepare_graph_data()

    # Step 4: Train GNN models
    print(f"\n{'='*70}")
    print("🧠 TRAINING GRAPH NEURAL NETWORKS")
    print(f"{'='*70}")

    gnn_system = GNNRoutingSystem(graph_data)
    training_losses = gnn_system.train_blockage_model(epochs=50)

    # Step 5: Predict blocked roads using GNN
    print(f"\n{'='*70}")
    print("🔮 GNN INFERENCE - PREDICTING ROAD BLOCKAGES")
    print(f"{'='*70}")

    blocked_indices, blockage_probs = gnn_system.predict_blocked_edges(threshold=0.5)

    # Show top risky roads
    top_k = 10
    top_risky_indices = torch.topk(blockage_probs, min(top_k, len(blockage_probs))).indices
    print(f"\n⚠️  Top {top_k} highest risk roads:")
    for rank, idx in enumerate(top_risky_indices, 1):
        print(f"   {rank}. Road segment {idx.item()} - Risk: {blockage_probs[idx]:.2%}")

    # Step 6: Get user location
    planner = GNNRoutePlanner(road_network, graph_data, blocked_indices)
    user_coords = planner.get_user_location()

    # Step 7: Find nearest shelter
    print(f"\n{'='*70}")
    print("🎯 FINDING OPTIMAL EVACUATION ROUTE")
    print(f"{'='*70}")

    shelter_info = planner.find_nearest_shelter(user_coords, shelters)

    if shelter_info is None:
        print("\n❌ Unable to find evacuation route")
        print("🚨 EMERGENCY: Contact local authorities immediately!")
        print("   Emergency Number: 112 (India)")
        return

    # Step 8: Calculate route
    route, distance, time, G_modified = planner.calculate_route(user_coords, shelter_info)

    if route is None:
        print("\n❌ Unable to calculate route")
        print("🚨 EMERGENCY: Contact local authorities immediately!")
        return

    # Step 9: Visualize
    print(f"\n{'='*70}")
    print("🗺️  GENERATING INTERACTIVE MAP")
    print(f"{'='*70}")

    visualize_route(G_modified, route, user_coords, shelter_info,
                   blocked_indices, shelters)

    # Step 10: Final summary
    print(f"\n{'='*70}")
    print("✅ GNN-BASED EVACUATION ROUTE CALCULATED")
    print(f"{'='*70}")

    if len(disasters) > 0:
        print(f"\n🚨 Active Disaster:")
        d = disasters[0]
        print(f"   Type: {d['type'].upper()}")
        print(f"   Severity: {d['severity'].upper()}")
        print(f"   Intensity: {d['intensity_score']:.1f}/10")

    print(f"\n📍 Evacuation Details:")
    print(f"   From: ({user_coords[0]:.4f}, {user_coords[1]:.4f})")
    print(f"   To: {shelter_info['name']}")
    print(f"   Distance: {distance/1000:.2f} km")
    print(f"   Est. Time: {time:.0f} minutes")

    print(f"\n🧠 GNN Analysis:")
    print(f"   Total road segments analyzed: {len(blockage_probs)}")
    print(f"   Predicted blocked: {len(blocked_indices)}")
    print(f"   Blockage rate: {len(blocked_indices)/len(blockage_probs)*100:.1f}%")
    print(f"   Average blockage risk: {blockage_probs.mean():.1%}")
    print(f"   Max blockage risk: {blockage_probs.max():.1%}")

    print(f"\n🛣️  Route Safety:")
    blocked_set = set(blocked_indices.numpy())
    edge_list = list(road_network.edges())
    route_edges = [(route[i], route[i+1]) for i in range(len(route)-1)]

    route_safe = True
    for u, v in route_edges:
        # Check if this edge is in blocked set
        for idx, (eu, ev) in enumerate(edge_list):
            if (u == eu and v == ev) or (u == ev and v == eu):
                if idx in blocked_set:
                    route_safe = False
                    break
        if not route_safe:
            break

    if route_safe:
        print(f"   ✅ Route avoids all predicted blockages")
    else:
        print(f"   ⚠️  Route may have minor risk - proceed with caution")

    print(f"\n🗺️  Interactive map saved as 'gnn_disaster_route_map.html'")
    print(f"\n💡 Safety Tips:")
    print(f"   • Follow official evacuation orders")
    print(f"   • Keep emergency supplies ready")
    print(f"   • Stay informed through local authorities")
    print(f"   • Emergency Number: 112")

    print(f"\n💾 Saving trained model...")
    torch.save({
        'model_state_dict': gnn_system.blockage_model.state_dict(),
        'num_node_features': graph_data.x.shape[1],
        'training_location': TARGET_CITY,
        'training_disaster': disasters[0]['type'] if len(disasters) > 0 else 'unknown'
    }, 'trained_model.pth')

    print("✅ Model saved!")
    
    print(f"\n{'='*70}")
    print("🚀 System Ready - Stay Safe!")
    print(f"{'='*70}\n")

# ===========================================
# EXECUTION
# ===========================================
if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️  Operation cancelled by user")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("Please check your internet connection and try again.")

🧠 GNN-BASED DISASTER RESPONSE ROUTING SYSTEM

🔍 Scanning for active disasters near Rudraprayag, Uttarakhand, India...
⚠️  ACTIVE DISASTERS DETECTED: 1
   • LANDSLIDE - Severity: high
     Intensity: 5.6/10
     Source: Heavy rainfall on steep terrain

📍 Loading geographic data for Rudraprayag, Uttarakhand, India...
✅ Loaded 298 nodes, 660 edges
✅ Loaded 352 water features
✅ Loaded 10 potential shelters

🔧 PREPARING GRAPH DATA FOR GNN

🔄 Converting to GNN-compatible graph structure...
✅ Graph prepared: 298 nodes, 1320 edges
   Node features: 6 dimensions
   Edge features: 7 dimensions
   Edge index range: [0, 297]

🧠 TRAINING GRAPH NEURAL NETWORKS

🧠 Training Road Blockage Prediction GNN...
   Epoch 10/50 | Loss: 0.5488 | Acc: 0.7697
   Epoch 20/50 | Loss: 0.5345 | Acc: 0.7697
   Epoch 30/50 | Loss: 0.5367 | Acc: 0.7697
   Epoch 40/50 | Loss: 0.5313 | Acc: 0.7697
   Epoch 50/50 | Loss: 0.5367 | Acc: 0.7697
✅ Blockage model trained

🔮 GNN INFERENCE - PREDICTING ROAD BLOCKAGES

🚧 Predicte


Select option (1/2/3):  3



🎯 FINDING OPTIMAL EVACUATION ROUTE

🏥 Finding nearest shelter...
🔄 Modified network: 660 edges remaining
✅ Nearest shelter: Government Hospital, Basukedar
   Distance: 39450 meters (39.45 km)

🛣️  Calculating GNN-optimized route...
✅ Route calculated
   Distance: 39.45 km
   Est. time: 78.9 minutes
   Segments: 17

🗺️  GENERATING INTERACTIVE MAP

🗺️  Creating interactive map...
✅ Map saved as 'gnn_disaster_route_map.html'

✅ GNN-BASED EVACUATION ROUTE CALCULATED

🚨 Active Disaster:
   Type: LANDSLIDE
   Severity: HIGH
   Intensity: 5.6/10

📍 Evacuation Details:
   From: (30.4916, 79.0366)
   To: Government Hospital, Basukedar
   Distance: 39.45 km
   Est. Time: 79 minutes

🧠 GNN Analysis:
   Total road segments analyzed: 660
   Predicted blocked: 0
   Blockage rate: 0.0%
   Average blockage risk: 22.4%
   Max blockage risk: 45.4%

🛣️  Route Safety:
   ✅ Route avoids all predicted blockages

🗺️  Interactive map saved as 'gnn_disaster_route_map.html'

💡 Safety Tips:
   • Follow official

In [ ]:
#!/usr/bin/env python3
"""
TESTING CODE - Load trained model and test on different locations
"""

import geopandas as gpd
import osmnx as ox
import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ===========================================
# CONFIGURATION - EDIT THESE!
# ===========================================
MODEL_PATH = "trained_model.pth"
SAFE_DISTANCE_FROM_WATER = 500

# TEST LOCATIONS - CHANGE THESE!
TEST_LOCATIONS = [
    {
        'name': 'Rishikesh Flood',
        'location': 'Rishikesh, Uttarakhand, India',  # ~5000 edges, 2 mins max
        'disaster_type': 'flood',
        'severity': 'medium'
    }
]
np.random.seed(42)
torch.manual_seed(42)

# ===========================================
# GNN MODEL (Same as training)
# ===========================================
class RoadBlockageGNN(nn.Module):
    def __init__(self, num_node_features, hidden_channels=64):
        super(RoadBlockageGNN, self).__init__()
        self.conv1 = GATConv(num_node_features, hidden_channels, heads=4, dropout=0.3)
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels, heads=4, dropout=0.3)
        self.conv3 = GATConv(hidden_channels * 4, hidden_channels, heads=2, dropout=0.3)
        
        self.edge_predictor = nn.Sequential(
            nn.Linear(hidden_channels * 2 * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.elu(self.conv1(x, edge_index))
        x = F.elu(self.conv2(x, edge_index))
        x = F.elu(self.conv3(x, edge_index))
        self.node_embeddings = x
        return x

    def predict_edge_blockage(self, edge_index):
        src_embeddings = self.node_embeddings[edge_index[0]]
        dst_embeddings = self.node_embeddings[edge_index[1]]
        edge_embeddings = torch.cat([src_embeddings, dst_embeddings], dim=1)
        blockage_prob = self.edge_predictor(edge_embeddings)
        return blockage_prob.squeeze()

# ===========================================
# DATA PREP (Same as training)
# ===========================================
class GraphDataPreparator:
    def __init__(self, road_network, water_bodies, disasters):
        self.G_nx = road_network
        self.water_bodies = water_bodies
        self.disasters = disasters
        self.scaler = StandardScaler()
        self.node_to_idx = {node: idx for idx, node in enumerate(self.G_nx.nodes())}
        self.idx_to_node = {idx: node for node, idx in self.node_to_idx.items()}

    def prepare_graph_data(self):
        node_features = self._extract_node_features()
        edge_features = self._extract_edge_features()
        edge_labels = self._generate_edge_labels()

        edges = []
        for u, v in self.G_nx.edges():
            u_idx = self.node_to_idx[u]
            v_idx = self.node_to_idx[v]
            edges.append([u_idx, v_idx])

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_features = torch.cat([edge_features, edge_features], dim=0)
        edge_labels = torch.cat([edge_labels, edge_labels], dim=0)

        data = Data(
            x=node_features,
            edge_index=edge_index,
            edge_attr=edge_features,
            edge_labels=edge_labels,
            num_nodes=len(self.G_nx.nodes())
        )
        return data

    def _extract_node_features(self):
        features = []
        nodes_gdf = ox.graph_to_gdfs(self.G_nx, edges=False, nodes=True)
        nodes_gdf = nodes_gdf.to_crs(epsg=3857)

        for node_id in self.G_nx.nodes():
            node_data = self.G_nx.nodes[node_id]
            node_features = []

            node_features.append(node_data['y'])
            node_features.append(node_data['x'])
            node_features.append(self.G_nx.degree(node_id))

            if len(self.water_bodies) > 0:
                try:
                    water_gdf = self.water_bodies.to_crs(epsg=3857)
                    node_point = nodes_gdf.loc[node_id, 'geometry']
                    min_water_dist = water_gdf.distance(node_point).min()
                    node_features.append(min_water_dist)
                except:
                    node_features.append(5000)
            else:
                node_features.append(5000)

            if len(self.disasters) > 0:
                epicenter = self.disasters[0]['epicenter']
                if epicenter:
                    from geopy.distance import geodesic
                    dist = geodesic((node_data['y'], node_data['x']), epicenter).meters
                    node_features.append(dist)
                else:
                    node_features.append(10000)
            else:
                node_features.append(10000)

            node_features.append(np.random.uniform(500, 2000))
            features.append(node_features)

        features_array = np.array(features)
        features_normalized = self.scaler.fit_transform(features_array)
        return torch.tensor(features_normalized, dtype=torch.float)

    def _extract_edge_features(self):
        features = []
        edges_gdf = ox.graph_to_gdfs(self.G_nx, nodes=False, edges=True)
        edges_gdf = edges_gdf.to_crs(epsg=3857)

        for u, v, data in self.G_nx.edges(data=True):
            edge_features = []
            edge_features.append(data.get('length', 100))

            highway_type = data.get('highway', 'residential')
            if isinstance(highway_type, list):
                highway_type = highway_type[0]
            road_type_encoding = {
                'motorway': 5, 'trunk': 4, 'primary': 3,
                'secondary': 2, 'tertiary': 1, 'residential': 0
            }
            edge_features.append(road_type_encoding.get(highway_type, 0))

            lanes = data.get('lanes', 2)
            if isinstance(lanes, list):
                lanes = int(lanes[0]) if lanes[0].isdigit() else 2
            elif isinstance(lanes, str):
                lanes = int(lanes) if lanes.isdigit() else 2
            edge_features.append(lanes)

            if len(self.water_bodies) > 0:
                try:
                    water_gdf = self.water_bodies.to_crs(epsg=3857)
                    edge_geom = edges_gdf.loc[(u, v, 0), 'geometry']
                    min_water_dist = water_gdf.distance(edge_geom).min()
                    edge_features.append(min_water_dist)
                except:
                    edge_features.append(5000)
            else:
                edge_features.append(5000)

            if len(self.disasters) > 0:
                disaster = self.disasters[0]
                edge_features.append(disaster['intensity_score'])
                disaster_encoding = {'flood': 0, 'earthquake': 1, 'landslide': 2}
                edge_features.append(disaster_encoding.get(disaster['type'], 0))
                edge_features.append(disaster['time_since_start_hours'])
            else:
                edge_features.extend([0, 0, 0])

            features.append(edge_features)
        return torch.tensor(features, dtype=torch.float)

    def _generate_edge_labels(self):
        labels = []
        edges_gdf = ox.graph_to_gdfs(self.G_nx, nodes=False, edges=True)
        edges_gdf = edges_gdf.to_crs(epsg=3857)

        for u, v, data in self.G_nx.edges(data=True):
            is_blocked = 0

            if len(self.disasters) > 0:
                disaster = self.disasters[0]

                if disaster['type'] == 'flood':
                    if len(self.water_bodies) > 0:
                        try:
                            water_gdf = self.water_bodies.to_crs(epsg=3857)
                            edge_geom = edges_gdf.loc[(u, v, 0), 'geometry']
                            min_water_dist = water_gdf.distance(edge_geom).min()

                            if disaster['severity'] == 'high' and min_water_dist < SAFE_DISTANCE_FROM_WATER * 2:
                                is_blocked = 1
                            elif disaster['severity'] == 'medium' and min_water_dist < SAFE_DISTANCE_FROM_WATER:
                                is_blocked = 1
                            elif disaster['severity'] == 'low' and min_water_dist < SAFE_DISTANCE_FROM_WATER * 0.5:
                                is_blocked = 1
                        except:
                            pass

                elif disaster['type'] == 'earthquake':
                    severity_prob = {'low': 0.05, 'medium': 0.15, 'high': 0.30}
                    if np.random.random() < severity_prob.get(disaster['severity'], 0.1):
                        is_blocked = 1

                elif disaster['type'] == 'landslide':
                    if np.random.random() < 0.25:
                        is_blocked = 1

            labels.append(is_blocked)
        return torch.tensor(labels, dtype=torch.float)

# ===========================================
# LOAD MODEL
# ===========================================
def load_model():
    print(f"📂 Loading model from {MODEL_PATH}...")
    checkpoint = torch.load(MODEL_PATH, weights_only=False)
    
    model = RoadBlockageGNN(num_node_features=checkpoint['num_node_features'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print(f"✅ Model loaded!")
    print(f"   Training location: {checkpoint.get('training_location', 'Unknown')}")
    print(f"   Training disaster: {checkpoint.get('training_disaster', 'Unknown')}")
    return model

# ===========================================
# TEST FUNCTION
# ===========================================
def test_location(model, location, disaster_type, severity):
    print(f"\n{'='*70}")
    print(f"🔬 TESTING: {location}")
    print(f"{'='*70}")
    
    # Create disaster info
    geolocator = Nominatim(user_agent="disaster_tester")
    try:
        loc = geolocator.geocode(location)
        epicenter = (loc.latitude, loc.longitude)
    except:
        epicenter = None
    
    disasters = [{
        'type': disaster_type,
        'severity': severity,
        'epicenter': epicenter,
        'intensity_score': 6.5,
        'time_since_start_hours': 6.0
    }]
    
    # Load data
    print(f"📍 Loading data...")
    road_network = ox.graph_from_place(location, network_type='drive', simplify=True)
    print(f"✅ Roads: {len(road_network.nodes)} nodes, {len(road_network.edges)} edges")
    
    try:
        water_bodies = ox.features.features_from_place(
            location,
            tags={'natural': ['water', 'waterway'], 'waterway': ['river', 'stream']}
        )
        print(f"✅ Water: {len(water_bodies)} features")
    except:
        print(f"⚠️  No water")
        water_bodies = gpd.GeoDataFrame()
    
    # Prepare graph
    print(f"🔄 Preparing graph...")
    data_prep = GraphDataPreparator(road_network, water_bodies, disasters)
    test_data = data_prep.prepare_graph_data()
    
    # Run inference
    print(f"🔮 Running inference...")
    model.eval()
    with torch.no_grad():
        _ = model(test_data)
        predictions = model.predict_edge_blockage(test_data.edge_index)
        predictions_binary = (predictions > 0.5).float()
    
    # Calculate metrics
    y_true = test_data.edge_labels.numpy()
    y_pred = predictions_binary.numpy()
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"\n📊 RESULTS:")
    print(f"   Accuracy:  {accuracy:.2%}")
    print(f"   Precision: {precision:.2%}")
    print(f"   Recall:    {recall:.2%}")
    print(f"   F1 Score:  {f1:.2%}")
    print(f"   TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")
    
    return {
        'location': location,
        'disaster': disaster_type,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': int(tp), 'tn': int(tn),
        'fp': int(fp), 'fn': int(fn)
    }

# ===========================================
# MAIN
# ===========================================
def main():
    print("="*70)
    print("🔬 TESTING SCRIPT")
    print("="*70)
    
    # Load model
    model = load_model()
    
    # Test all locations
    results = []
    for test in TEST_LOCATIONS:
        try:
            result = test_location(
                model,
                test['location'],
                test['disaster_type'],
                test['severity']
            )
            result['name'] = test['name']
            results.append(result)
        except Exception as e:
            print(f"❌ Failed: {e}")
    
    # Summary
    if results:
        print(f"\n{'='*70}")
        print("📊 SUMMARY")
        print(f"{'='*70}")
        
        df = pd.DataFrame(results)
        print(f"\nTotal: {len(results)} tests")
        print(f"Avg Accuracy: {df['accuracy'].mean():.2%}")
        print(f"Avg F1: {df['f1'].mean():.2%}")
        
        print(f"\n{'Name':<30} {'Disaster':<12} {'Accuracy':<12} {'F1':<12}")
        print("-" * 70)
        for _, row in df.iterrows():
            print(f"{row['name']:<30} {row['disaster']:<12} {row['accuracy']:<12.2%} {row['f1']:<12.2%}")
        
        df.to_csv('test_results.csv', index=False)
        print(f"\n✅ Saved to test_results.csv")
    
    print(f"\n{'='*70}")
    print("✅ DONE!")
    print(f"{'='*70}\n")

if __name__ == "__main__":
    main()

🔬 TESTING SCRIPT
📂 Loading model from trained_model.pth...
✅ Model loaded!
   Training location: Rudraprayag, Uttarakhand, India
   Training disaster: landslide

🔬 TESTING: Rishikesh, Uttarakhand, India
📍 Loading data...
✅ Roads: 9643 nodes, 22387 edges
✅ Water: 131 features
🔄 Preparing graph...
